# RoSE on ChartQAPro — **all factoid questions**, headless Kaggle commit

Runs the repo's real pipeline (`chartqapro/rose_chartqapro.py` + `scoring.py`) over
**every Factoid question in ChartQAPro**, then evaluates the run offline with
`ablate.py`, `summarize.py` and `export_csv.py` — so the notebook, the CLI tables and
the web dashboard all report the same numbers.

Sibling of `RoSE_ChartQAPro_MCQ_Kaggle.ipynb`; the two share every cell except the
dataset prep, the pre-flight check and the diagnostics.

| | |
|---|---|
| **Dataset** | [`ahmed-masry/ChartQAPro`](https://huggingface.co/datasets/ahmed-masry/ChartQAPro) — `Factoid` only |
| **Model** | `Qwen/Qwen2.5-VL-7B-Instruct`, 4-bit |
| **Pipeline** | cloned from the repo at `REPO_REF` — no algorithm code is re-implemented here |
| **Scoring** | `scoring.py` only — normalized exact match **plus 5% numeric tolerance** |

## What differs from the MCQ run

Factoid answers are open-ended, which changes what matters:

- **No options.** Nothing to permute, so `mcq_permute_options` and
  `mcq_demo_anchor_fix` are inert here; the vote is over free-text spans.
- **`numeric_vote_clustering` earns its keep.** Paths that read `45.2` and `45` off
  the same chart are the same answer. This is the one extension the offline ablation
  in Cell 10 can actually move on factoid — on MCQ it is a no-op.
- **Unanswerable is a real class.** ChartQAPro factoid includes questions the chart
  cannot answer, so the punt rate in both directions is reported in Cells 9 and 13.
- **It is the biggest slice of the dataset**, so the 12h wall is the binding
  constraint — see sharding below.

## Run it headless on Kaggle

1. **Settings** → Accelerator **GPU T4 ×2** (or P100), Internet **ON**, Persistence *off*.
2. **Save Version → Save & Run All (Commit)**. Every cell is non-interactive and
   failure-tolerant, so the commit finishes and publishes its outputs even if the
   GPU wall is hit mid-run.
3. `TIME_BUDGET_MIN` (Cell 1) stops the loop *before* Kaggle kills the session and
   writes a complete checkpoint. `meta_factoid.json` records `"complete": false` when
   the run was cut short.

## Finishing a big run: shard or resume

**Resume** — add the finished version's output as a dataset input
(**+ Add Data → Your Work → this notebook's output**) and set in Cell 1:

```python
RESUME_FROM = "/kaggle/input/<your-notebook-output>/results/checkpoint_factoid.json"
```

**Shard** — run several commits in parallel, each taking every Nth question:

```python
SHARD, SHARDS = 0, 3      # then 1,3 and 2,3 in two more commits
```

Sharding is faster but splits the experience pool three ways, so each shard's pool
grows to a third the size. Resuming keeps one continuous pool and is the honest
setting for a learning-curve claim; shard when you only need the accuracy number.

## Outputs (`/kaggle/working/results/`)

| File | What it is |
|---|---|
| `checkpoint_factoid.json` | resume point, rewritten every `CHECKPOINT_EVERY` questions |
| `rose_factoid.json` | the run — one record per question, **including every sampled path** |
| `meta_factoid.json` | model, hyperparameters, active extensions, commit SHA, completeness |
| `summary_factoid.json` | `summarize.py` output — what the dashboard reads |
| `rose_factoid.csv` / `rose_factoid_per_path.csv` | `export_csv.py` — per question / per reasoning path |
| `factoid_diagnostics.png` | learning curve + entropy calibration |

---
## Cell 1 — CONFIG · **the only cell you edit**

In [ ]:
from pathlib import Path
import os, sys

# Kaggle's log pane shows whatever has been flushed, and rose_chartqapro.py's
# own prints do not pass flush=True. Line-buffering stdout once here makes every
# downstream print stream out instead of arriving in blocks.
try:
    sys.stdout.reconfigure(line_buffering=True)
except (AttributeError, ValueError):
    pass

# ── where the pipeline code comes from ───────────────────────────
REPO_URL   = "https://github.com/ktahsinr/RoSE.git"
REPO_REF   = "fix/scoring-retrieval-and-extensions"   # branch, tag or SHA
LOCAL_CODE = None      # set to a /kaggle/input/... path holding chartqapro/*.py
                       # if the notebook must run with Internet OFF

# ── what to run ──────────────────────────────────────────────────
SPLIT        = "test"          # ChartQAPro split
LIMIT        = None            # None = ALL Factoid questions (this is the point)
SHARD, SHARDS = 0, 1           # split the set across parallel commits: SHARD=1 SHARDS=3

# ── model + RoSE hyperparameters ─────────────────────────────────
MODEL_NAME       = "Qwen/Qwen2.5-VL-7B-Instruct"
M_PATHS          = 5      # paper: 20 — 5 is the T4 compromise
K_DEMONSTRATIONS = 3
LAMBDA           = 1.2
TEMPERATURE      = 0.8
MAX_NEW_TOKENS   = 300
VISUAL_ALPHA     = 0.35

# ── extensions (see README table) ────────────────────────────────
PAPER_FAITHFUL      = False    # True = disable all ten, for the baseline row
EXTENSION_OVERRIDES = {}       # e.g. {"numeric_vote_clustering": False} for one ablation run

# ── headless safety ──────────────────────────────────────────────
TIME_BUDGET_MIN  = 11 * 60     # stop + save before Kaggle's 12h commit wall
CHECKPOINT_EVERY = 10
RESUME_FROM      = None        # path to a previous commit's checkpoint_factoid.json

# ── paths ────────────────────────────────────────────────────────
ON_KAGGLE   = Path("/kaggle").exists()
WORK        = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
SCRATCH     = Path("/kaggle/temp")    if ON_KAGGLE else WORK / "scratch"   # not published
CODE_DIR    = SCRATCH / "RoSE"
IMAGES_DIR  = SCRATCH / "images"      # regenerated each run; never counts against output quota
RESULTS_DIR = WORK / "results"
DATASET_JSON    = SCRATCH / "chartqapro_factoid.json"
CHECKPOINT_FILE = RESULTS_DIR / "checkpoint_factoid.json"
FINAL_FILE      = RESULTS_DIR / "rose_factoid.json"      # summarize.py maps this to meta_factoid.json
META_FILE       = RESULTS_DIR / "meta_factoid.json"

for d in (SCRATCH, IMAGES_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"on kaggle    : {ON_KAGGLE}")
print(f"results dir  : {RESULTS_DIR}")
print(f"time budget  : {TIME_BUDGET_MIN} min")
print(f"resume from  : {RESUME_FROM}")

## Cell 2 — Dependencies + GPU

`Qwen2_5_VLForConditionalGeneration` only exists in `transformers >= 4.49`; on older
versions `rose_chartqapro.py` silently falls back to the Qwen2-VL class, which
mis-handles some 2.5 checkpoints. The install below is therefore version-checked,
not just presence-checked.

In [ ]:
import importlib, subprocess, sys

need = []
try:
    from transformers import Qwen2_5_VLForConditionalGeneration  # noqa: F401
except Exception:
    need.append("transformers==4.51.3")
for pkg, spec in [("accelerate", "accelerate>=0.34"),
                  ("bitsandbytes", "bitsandbytes>=0.43"),
                  ("sentence_transformers", "sentence-transformers>=3.0"),
                  ("datasets", "datasets>=2.19")]:
    if importlib.util.find_spec(pkg) is None:
        need.append(spec)

if need:
    print("installing:", need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=False)
    print("✓ installed — restart is NOT required on Kaggle images")
else:
    print("✓ all dependencies present")

import torch, transformers
print(f"\ntorch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}        : {p.name}  {p.total_memory/1e9:.1f} GB")
else:
    print("⚠  NO GPU — inference cells will fail. Set Accelerator to GPU T4 x2.")

## Cell 3 — Get the pipeline code

The RoSE algorithm, the prompts and every scoring rule live in the repo. Cloning
them (rather than pasting them here) is what keeps this notebook's numbers identical
to `python ablate.py` on a laptop. The commit SHA is recorded in `meta_factoid.json`.

In [ ]:
import subprocess, sys, shutil, importlib

def _sh(*args, cwd=None):
    return subprocess.run(args, cwd=cwd, capture_output=True, text=True)

CQA_DIR = None

if LOCAL_CODE:                                   # Internet OFF path
    src = Path(LOCAL_CODE)
    CQA_DIR = src if (src / "rose_chartqapro.py").exists() else src / "chartqapro"
    COMMIT_SHA = "local-copy"
else:
    if CODE_DIR.exists():
        shutil.rmtree(CODE_DIR)
    r = _sh("git", "clone", "--depth", "50", "--branch", REPO_REF, REPO_URL, str(CODE_DIR))
    if r.returncode != 0:                        # REPO_REF may be a bare SHA
        _sh("git", "clone", "--depth", "50", REPO_URL, str(CODE_DIR))
        _sh("git", "checkout", REPO_REF, cwd=str(CODE_DIR))
    CQA_DIR = CODE_DIR / "chartqapro"
    COMMIT_SHA = _sh("git", "rev-parse", "HEAD", cwd=str(CODE_DIR)).stdout.strip() or "unknown"

assert CQA_DIR and (CQA_DIR / "rose_chartqapro.py").exists(), (
    f"pipeline code not found at {CQA_DIR}. Either enable Internet (to clone "
    f"{REPO_URL}) or set LOCAL_CODE to a Kaggle dataset holding chartqapro/*.py")

sys.path.insert(0, str(CQA_DIR))

# The clone can only contain what was COMMITTED AND PUSHED to REPO_REF. A file
# that is merely present in the local working tree is not here — so the imports
# are split into what the run needs and what only the evaluation needs, and the
# gap is reported now rather than after ten GPU-hours.
REQUIRED = ["rose_chartqapro", "scoring"]            # inference + verdicts
OPTIONAL = ["ablate", "summarize", "export_csv"]     # offline evaluation only

mods, missing = {}, []
for name in REQUIRED + OPTIONAL:
    try:
        mods[name] = importlib.import_module(name)
    except Exception as exc:
        mods[name] = None
        missing.append((name, exc))

rose, scoring = mods["rose_chartqapro"], mods["scoring"]
ablate, summarize, export_csv = mods["ablate"], mods["summarize"], mods["export_csv"]

assert rose and scoring, (
    f"the run itself cannot start: {[n for n, _ in missing if n in REQUIRED]} "
    f"missing from {CQA_DIR}")

# summarize.py does `from ablate import ablation_table`; an older ablate.py
# without it breaks both.
STALE_ABLATE = ablate is not None and not hasattr(ablate, "ablation_table")

print(f"✓ pipeline   : {CQA_DIR}")
print(f"✓ commit     : {COMMIT_SHA[:12]}  ({REPO_REF})")
print(f"✓ required   : rose_chartqapro, scoring")
print(f"  evaluation : " + ", ".join(n for n in OPTIONAL if mods[n]) or "  evaluation : none")

if missing or STALE_ABLATE:
    print("\n" + "!" * 70)
    for n, exc in missing:
        print(f"  missing: {n}.py  ({type(exc).__name__})")
    if STALE_ABLATE:
        print("  stale  : ablate.py has no ablation_table() — Cells 10-12 need the newer one")
    print(f"\n  These files exist in your working tree but are NOT on '{REPO_REF}'.")
    print("  Inference is unaffected; the offline evaluation cells will skip.")
    print("  To fix, from the repo root:\n")
    print("      git add chartqapro/summarize.py chartqapro/ablate.py")
    print('      git commit -m "Add summarize.py + ablation_table for the Kaggle run"')
    print(f"      git push origin {REPO_REF}\n")
    print("  then re-run this cell. Or set LOCAL_CODE to a Kaggle dataset holding")
    print("  chartqapro/*.py uploaded straight from your laptop.")
    print("!" * 70)

## Cell 4 — Build the factoid set

Simpler than the MCQ prep — there are no options to recover, so a record is just the
question, the gold answer and the chart image. The gold is kept in its original list
form; `scoring.split_gold` treats every element as an acceptable answer, which is how
ChartQAPro expresses "either of these is right".

In [ ]:
import ast, io, json, re
from collections import Counter
from PIL import Image

FACTOID_LABELS = {"factoid"}


def _gold(v):
    """Keep multi-answer gold as a list; scoring.split_gold accepts either."""
    if isinstance(v, (list, tuple)):
        return [str(x) for x in v]
    s = str(v).strip()
    if s.startswith("["):
        try:
            p = ast.literal_eval(s)
            if isinstance(p, (list, tuple)):
                return [str(x) for x in p]
        except Exception:
            pass
    return s


def _save_image(field, path):
    if isinstance(field, dict) and field.get("bytes"):
        img = Image.open(io.BytesIO(field["bytes"]))
    elif isinstance(field, (bytes, bytearray)):
        img = Image.open(io.BytesIO(field))
    elif isinstance(field, Image.Image):
        img = field
    else:                                    # already a path on disk
        return str(field)
    img.convert("RGB").save(path, format="PNG")
    return str(path)


# ── load + filter ────────────────────────────────────────────────
from datasets import load_dataset

ds = load_dataset("ahmed-masry/ChartQAPro", split=SPLIT)
print(f"{SPLIT} split: {len(ds)} questions")
print("by type:", dict(Counter(str(t) for t in ds["Question Type"]).most_common()))

# Select by column first, so only the images this run actually needs get decoded
# and written — the rest of the split is never touched.
qtypes  = [str(t).strip().lower() for t in ds["Question Type"]]
fact_idx = [i for i, t in enumerate(qtypes) if t in FACTOID_LABELS]
if SHARDS > 1:
    fact_idx = fact_idx[SHARD::SHARDS]
if LIMIT:
    fact_idx = fact_idx[:LIMIT]
print(f"factoid rows selected: {len(fact_idx)}")

records = []
for idx, i in enumerate(fact_idx):
    row = ds[i]
    records.append({
        "id":            str(row.get("id", f"factoid-{idx:05d}")),
        "question":      str(row.get("Question", "")).strip(),
        "choices":       None,                        # open-ended
        "answer":        _gold(row.get("Answer")),
        "question_type": "factoid",                   # the label the pipeline expects
        "image":         _save_image(row.get("image"), IMAGES_DIR / f"{idx:05d}.png"),
        "orig_type":     str(row.get("Question Type", "")),
        "year":          row.get("Year"),
    })

DATASET_JSON.write_text(json.dumps(records, ensure_ascii=False, indent=2))

print(f"\nfactoid questions to run : {len(records)}")
print(f"dataset json             : {DATASET_JSON}")
print(f"images                   : {IMAGES_DIR}")

## Cell 5 — Pre-flight check

Open-ended gold has failure modes an MCQ run cannot have: an answer that normalizes to
the empty string can never be matched by anything, and the answerable/unanswerable
split decides how much of the score the punt behaviour controls. Both are measured
here with the same `scoring.py` functions that will score the run, so a bad batch
shows up now rather than after the GPU hours.

In [ ]:
n_unans = n_numeric = n_multi = n_empty = 0
for r in records:
    golds = scoring.split_gold(r["answer"])
    if not golds:
        n_empty += 1
        continue
    if len(golds) > 1:
        n_multi += 1
    if any(scoring.is_unanswerable(g) for g in golds):
        n_unans += 1
    elif any(scoring.number_variants(g) for g in golds):
        n_numeric += 1
    if not any(scoring.normalize_answer(g, "factoid") for g in golds):
        n_empty += 1

n = max(len(records), 1)
print(f"questions            : {len(records)}")
print(f"unanswerable gold    : {n_unans:5d}  ({n_unans/n*100:4.1f}%)  "
      f"— the punt class; see the confusion counts in Cell 9")
print(f"numeric gold         : {n_numeric:5d}  ({n_numeric/n*100:4.1f}%)  "
      f"— scored with {scoring.NUMERIC_TOLERANCE:.0%} tolerance")
print(f"multi-answer gold    : {n_multi:5d}  ({n_multi/n*100:4.1f}%)  "
      f"— any element counts as correct")
print(f"gold normalizes empty: {n_empty:5d}  "
      f"{'⚠  these can never be scored correct' if n_empty else '✓'}")

print()
for r in records[:3]:
    golds = scoring.split_gold(r["answer"])
    print("=" * 70)
    print("question :", r["question"][:200].replace("\n", " "))
    print("gold     :", r["answer"])
    print("normalized:", [scoring.normalize_answer(g, "factoid") for g in golds])
    print("image    :", r["image"])

## Cell 6 — Point the pipeline at this run

`rose_chartqapro.py` reads its configuration from module-level constants, so a run is
configured by overriding them — nothing is forked or copied.

In [ ]:
rose.MODEL_NAME       = MODEL_NAME
rose.TARGET_TYPES     = {"factoid"}
rose.M_PATHS          = M_PATHS
rose.K_DEMONSTRATIONS = K_DEMONSTRATIONS
rose.LAMBDA           = LAMBDA
rose.TEMPERATURE      = TEMPERATURE
rose.MAX_NEW_TOKENS   = MAX_NEW_TOKENS
rose.VISUAL_ALPHA     = VISUAL_ALPHA
rose.CHECKPOINT_EVERY = CHECKPOINT_EVERY

rose.DATASET_PATH     = str(DATASET_JSON)
rose.IMAGES_DIR       = str(IMAGES_DIR)
rose.RESULTS_DIR      = str(RESULTS_DIR)
rose.CHECKPOINT_FILE  = str(CHECKPOINT_FILE)
rose.FINAL_FILE       = str(FINAL_FILE)

rose.PAPER_FAITHFUL   = PAPER_FAITHFUL
rose.EXTENSIONS.update(EXTENSION_OVERRIDES)

print(f"model   : {rose.MODEL_NAME}")
print(f"types   : {sorted(rose.TARGET_TYPES)}")
print(f"m={rose.M_PATHS}  k={rose.K_DEMONSTRATIONS}  λ={rose.LAMBDA}  T={rose.TEMPERATURE}")
print(f"paper faithful : {rose.PAPER_FAITHFUL}")
for name, on in rose.active_extensions().items():
    print(f"   {'✓' if on else '·'} {name}")

## Cell 7 — The headless loop

Same body as `rose_chartqapro.run()` — same `process_one`, same warm-restart pool
rebuild, same checkpoint semantics — with three things a Kaggle commit needs:

- a **wall-clock budget** that stops cleanly instead of being killed mid-write,
- **one log line per question** (a 12h commit log stays readable),
- `meta_factoid.json` carrying **`complete`**, the commit SHA and the elapsed time, so a
  partial run can never be mistaken for a finished one.

In [ ]:
import time, traceback, json


def run_factoid_headless(data, model, processor, embedder, clip_model, clip_processor):
    t0 = time.time()
    budget_s = TIME_BUDGET_MIN * 60

    # ── resume ───────────────────────────────────────────────────
    results = []
    resume_src = None
    for cand in (CHECKPOINT_FILE, Path(RESUME_FROM) if RESUME_FROM else None):
        if cand and Path(cand).exists():
            results = json.loads(Path(cand).read_text())
            resume_src = cand
            break
    start_idx = len(results)
    if resume_src:
        print(f"↩  resuming at {start_idx}/{len(data)} from {resume_src}")

    # ── rebuild the pool from what is already answered ───────────
    pool = rose.ExperiencePool(embedder)
    for r in results:
        if r.get("method") == "error":
            continue
        pool.add(question=r["question"],
                 rationale=r.get("best_rationale") or r.get("prediction", ""),
                 answer=r.get("prediction", ""),
                 qtype=r.get("question_type", "factoid"),
                 uncertainty=r.get("uncertainty", 0.5),
                 complexity=r.get("complexity", 1.0),
                 clip_emb=None)          # embeddings are not checkpointed; text-only retrieval

    def save(final=False):
        CHECKPOINT_FILE.write_text(json.dumps(results, indent=2))
        if final:
            FINAL_FILE.write_text(json.dumps(results, indent=2))

    print(f"\n{'='*66}\n  RoSE · ChartQAPro Factoid · {len(data)-start_idx} to go"
          f"  |  m={rose.M_PATHS} k={rose.K_DEMONSTRATIONS}\n{'='*66}")

    complete = True
    try:
        for i, sample in enumerate(data[start_idx:], start=start_idx):
            if time.time() - t0 > budget_s:
                print(f"\n⏱  time budget reached at {i}/{len(data)} — saving and stopping cleanly")
                complete = False
                break
            try:
                r = rose.process_one(sample, model, processor, pool,
                                     clip_model=clip_model, clip_processor=clip_processor)
                results.append(r)
                print(f"[{i+1:04d}/{len(data)}] {'✓' if r['is_correct'] else '✗'} "
                      f"{r['method'][:9]:<9} pool={r['pool_size']:<4} "
                      f"pred={str(r['prediction'])[:22]:<22} gold={str(r['ground_truth'])[:22]:<22} "
                      f"u={r['uncertainty']:.2f}", flush=True)
            except Exception as exc:
                print(f"[{i+1:04d}/{len(data)}] ⚠  {exc}", flush=True)
                traceback.print_exc()
                results.append({"id": sample.get("id", ""), "question": sample.get("question", ""),
                                "question_type": "factoid", "ground_truth": sample.get("answer", ""),
                                "prediction": "ERROR", "is_correct": False,
                                "error": str(exc), "method": "error"})

            if (i + 1) % CHECKPOINT_EVERY == 0:
                save()
                ok = [x for x in results if x.get("method") != "error"]
                acc = sum(x.get("is_correct", False) for x in ok) / max(len(ok), 1) * 100
                print(f"  💾 checkpoint {i+1}/{len(data)}  running acc={acc:.1f}%  "
                      f"elapsed={(time.time()-t0)/60:.0f}m", flush=True)
    except KeyboardInterrupt:
        # Ctrl-C (and Kaggle's stop button) raises BaseException, which
        # the per-question handler deliberately does not catch. Without
        # this the function would unwind before writing its final files.
        print("\n⛔ interrupted — saving what is already answered", flush=True)
        complete = False

    save(final=True)

    META_FILE.write_text(json.dumps({
        "model": rose.MODEL_NAME, "embed_model": rose.EMBED_MODEL,
        "clip_model": "openai/clip-vit-base-patch32" if clip_model else None,
        "m_paths": rose.M_PATHS, "k_demonstrations": rose.K_DEMONSTRATIONS,
        "lambda": rose.LAMBDA, "temperature": rose.TEMPERATURE,
        "visual_alpha": rose.VISUAL_ALPHA, "max_new_tokens": rose.MAX_NEW_TOKENS,
        "numeric_tolerance": rose.NUMERIC_TOLERANCE,
        "offer_unanswerable": rose.OFFER_UNANSWERABLE,
        "target_types": ["factoid"], "paper_faithful": rose.PAPER_FAITHFUL,
        "extensions": rose.active_extensions(),
        "dataset": f"ahmed-masry/ChartQAPro:{SPLIT}", "shard": [SHARD, SHARDS],
        "code_commit": COMMIT_SHA, "code_ref": REPO_REF,
        "n_target": len(data), "n_results": len(results),
        "complete": complete and len(results) >= len(data),
        "elapsed_min": round((time.time() - t0) / 60, 1),
        "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }, indent=2))

    print(f"\n{'✓ COMPLETE' if complete else '◐ PARTIAL — rerun with RESUME_FROM'}"
          f"  {len(results)}/{len(data)}  →  {FINAL_FILE}")
    return results

## Cell 8 — Run it  ⏳

The long cell. Loads the VLM (4-bit), the sentence embedder and CLIP, then streams
every Factoid question through RoSE with the experience pool growing as it goes.

**Expect a long silence before the first line.** On a fresh session the model
weights are downloaded (several GB) before anything is printed, and the first
question then needs `M_PATHS + 1` generations. Minutes without output here is
normal; the per-question lines start once loading finishes.

**In a Save & Run All commit, Kaggle does not render cell output live** — the
notebook you see updates only when the version finishes. To watch a commit in
flight, open the version and read the **Logs** pane. Interactive sessions do
stream the output into the cell.


In [ ]:
import time

data = json.loads(DATASET_JSON.read_text())
print(f"[{time.strftime('%H:%M:%S')}] {len(data)} Factoid questions queued", flush=True)
print(f"[{time.strftime('%H:%M:%S')}] loading {rose.MODEL_NAME} — a fresh session "
      f"downloads the weights first, so this step is silent for several minutes",
      flush=True)

model, processor, embedder = rose.load_models()
clip_model, clip_processor = rose.load_clip()
print(f"[{time.strftime('%H:%M:%S')}] models ready — starting the sweep", flush=True)

results = run_factoid_headless(data, model, processor, embedder, clip_model, clip_processor)

---
# Evaluation

Everything below is CPU-only and reads `rose_factoid.json` from disk, so it can be rerun
on a laptop — or on a partial run, mid-sweep. Each cell no-ops with a message rather
than raising, so a commit still publishes its outputs when the run was cut short.

## Cell 9 — Headline accuracy

In [ ]:
import json
from pathlib import Path

# FINAL_FILE appears only once the sweep returns. A run that is still
# going — or was interrupted — has everything answered so far in the
# checkpoint, so evaluate whichever exists and say which one was used.
SOURCE = FINAL_FILE if Path(FINAL_FILE).exists() else CHECKPOINT_FILE

rows = json.loads(Path(SOURCE).read_text()) if Path(SOURCE).exists() else []
meta = json.loads(Path(META_FILE).read_text()) if Path(META_FILE).exists() else {}

if not rows:
    print(f"no results yet — run Cell 8, or wait for its first checkpoint "
          f"(written every {CHECKPOINT_EVERY} questions)")
else:
    partial = SOURCE != FINAL_FILE
    print(f"source : {Path(SOURCE).name}"
          + ("   ← PARTIAL, straight from the checkpoint" if partial else ""))
    print(f"rows   : {len(rows)}")
    if meta:
        print(f"run    : {meta.get('n_results')}/{meta.get('n_target')}   "
              f"complete={meta.get('complete')}   "
              f"commit={str(meta.get('code_commit'))[:12]}")
    rose.print_results_table(rows)


## Cell 10 — Offline aggregation ablation

`numeric_vote_clustering`, `drop_malformed_paths` and `confidence_weighted_vote` only
change how the *m* logged per-path answers are combined, so all four configurations
are recomputed from this one run — no GPU. The other seven extensions change the
prompt, the decoding or the pool, so each needs its own commit with the flag flipped
in `EXTENSION_OVERRIDES`.

In [ ]:
if not rows:
    print("no results yet")
elif ablate is None or STALE_ABLATE:
    print("skipped — ablate.ablation_table() is not in this clone (see Cell 3).\n"
          "The per-path logs are in rose_factoid.json, so this runs later on a laptop:\n"
          f"    python ablate.py {Path(SOURCE).name} --by-type")
else:
    for block in ablate.ablation_table(rows, by_type=True):
        print(f"\n{'='*66}\n  {block['label']}   (n={block['n']})\n{'='*66}")
        for c in block["configs"]:
            d = f"{c['delta']:+.1f}" if c.get("delta") is not None else "  base"
            print(f"  {c['name']:<34} {c['acc']:5.1f}%  ({c['correct']}/{c['total']})  {d}")
        if block.get("skipped"):
            print(f"  ({block['skipped']} rows had no per-path log and used their stored prediction)")

## Cell 11 — `summarize.py` + CSV export

`summary_factoid.json` is exactly what the web dashboard reads, and the CSVs are what goes
into a results table. Both re-score every row through `scoring.py` on the way out, so
they carry the current verdict rather than whatever flag was written during the run.

In [ ]:
import subprocess, sys

summary = None
if not rows:
    print("no results yet")
elif summarize is None or STALE_ABLATE:
    print("skipped — summarize.py is not in this clone (see Cell 3).\n"
          f"    python summarize.py {Path(SOURCE).name}      # later, on a laptop")
else:
    summary = summarize.summarize(str(SOURCE))
    (RESULTS_DIR / "summary_factoid.json").write_text(json.dumps(summary, indent=2))

    o = summary["overall"]
    print(f"overall (rescored) : {o['rescored']['acc']:.1f}%  "
          f"({o['rescored']['correct']}/{o['rescored']['total']})")
    print(f"overall (as run)   : {o['run']['acc']:.1f}%   drift={o['drift']:+d}")
    print(f"errors             : {summary['n_errors']}")
    print("\nby method:")
    for b in summary["by_method"]:
        print(f"   {b['key']:<16} {b['acc']:5.1f}%  ({b['correct']}/{b['total']})")
    print("\nby vote mode:")
    for b in summary["by_vote_mode"]:
        print(f"   {b['key']:<16} {b['acc']:5.1f}%  ({b['correct']}/{b['total']})")
    print("\nanswerability confusion:", summary["confusion"])

# export_csv.py stands alone — it only needs scoring.py, so the CSVs are written
# even when the summarize/ablate pair is missing from the clone.
if rows and (CQA_DIR / "export_csv.py").exists():
    # --per-path writes both the per-question CSV and the per-path one
    r = subprocess.run([sys.executable, str(CQA_DIR / "export_csv.py"),
                        str(SOURCE), "--per-path"], capture_output=True, text=True)
    print("\n" + (r.stdout.strip() or r.stderr.strip()))

## Cell 12 — Does the pool actually help?

The RoSE claim in two plots: accuracy should climb as the experience pool grows, and
the entropy the pool is filtered by should be predictive of correctness — high
uncertainty bins should score worse than low ones. If the second plot is flat, λ is
gating on noise.

In [ ]:
import matplotlib.pyplot as plt

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
BLUE             = "#2a78d6"

if not rows or summary is None:
    print("skipped — needs the summary from Cell 11")
else:
    curve = summary["curve"]
    bins  = summary["uncertainty_bins"]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), facecolor="white")

    # ── learning curve ───────────────────────────────────────────
    xs = [p["i"] for p in curve["points"]]
    ys = [p["roll_acc"] for p in curve["points"]]
    ax1.plot(xs, ys, color=BLUE, lw=2, solid_capstyle="round")
    if curve["phase_switch"]:
        ax1.axvspan(0, curve["phase_switch"], color=GRID, alpha=0.7, lw=0)
        ax1.annotate("zero-shot (pool < k)", xy=(curve["phase_switch"], 1),
                     xycoords=("data", "axes fraction"), xytext=(4, -10),
                     textcoords="offset points", fontsize=8.5, color=MUTED, va="top")
    final = summary["overall"]["rescored"]["acc"]
    ax1.axhline(final, color=MUTED, lw=1, ls=(0, (4, 3)))
    ax1.annotate(f"{final:.1f}% overall", xy=(1, final), xycoords=("axes fraction", "data"),
                 xytext=(-4, 4), textcoords="offset points",
                 ha="right", va="bottom", fontsize=8.5, color=MUTED)
    ax1.set_title(f"Rolling accuracy as the pool grows  (window {curve['window']})",
                  fontsize=11, color=INK, loc="left")
    ax1.set_xlabel("question, in stream order", fontsize=9, color=MUTED)
    ax1.set_ylabel("accuracy %", fontsize=9, color=MUTED)

    # ── uncertainty calibration ──────────────────────────────────
    labels = [f"{b['lo']:.2f}–{b['hi']:.2f}" for b in bins]
    accs   = [b["acc"] for b in bins]
    ax2.bar(range(len(bins)), accs, color=BLUE, width=0.62)
    for i, b in enumerate(bins):
        ax2.text(i, b["acc"] + 2, f"{b['acc']:.0f}%\nn={b['total']}",
                 ha="center", va="bottom", fontsize=8.5, color=MUTED)
    ax2.set_xticks(range(len(bins)))
    ax2.set_xticklabels(labels, fontsize=8.5, color=MUTED)
    ax2.set_yticks([0, 25, 50, 75, 100])
    ax2.set_title("Accuracy by entropy bin  (low uncertainty → left)",
                  fontsize=11, color=INK, loc="left")
    ax2.set_xlabel("path-vote entropy", fontsize=9, color=MUTED)
    ax2.set_ylabel("accuracy %", fontsize=9, color=MUTED)
    ax2.set_ylim(0, 118)

    for ax in (ax1, ax2):
        ax.grid(axis="y", color=GRID, lw=0.8)
        ax.set_axisbelow(True)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        for s in ("left", "bottom"):
            ax.spines[s].set_color(GRID)
        ax.tick_params(colors=MUTED, labelsize=9)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "factoid_diagnostics.png", dpi=150, facecolor="white")
    plt.show()

## Cell 13 — Factoid-specific diagnostics

Four things an aggregate accuracy number hides on open-ended answers:

- **How much of the score the 5% numeric tolerance is carrying.** A large gap between
  exact and tolerance-credited means the model reads charts approximately — true, but
  worth stating rather than burying.
- **Where the errors sit** — numeric gold vs text gold vs unanswerable.
- **Malformed paths**: how often a sampled path never emitted `Final Answer:`. This is
  what `drop_malformed_paths` acts on, so it sizes that extension's headroom.
- **How often the vote actually clustered** rather than matching exactly — the direct
  evidence for `numeric_vote_clustering`.

In [ ]:
from collections import Counter

if not rows:
    print("no results yet")
else:
    valid = [r for r in rows if r.get("method") != "error"]

    # ── what the numeric tolerance is worth ──────────────────────
    exact = tol = 0
    for r in valid:
        if not r.get("is_correct"):
            continue
        pn = scoring.normalize_answer(r.get("prediction", ""), "factoid")
        if any(pn == scoring.normalize_answer(g, "factoid")
               for g in scoring.split_gold(r.get("ground_truth"))):
            exact += 1
        else:
            tol += 1
    n = max(len(valid), 1)
    print(f"correct, exact normalized match : {exact:5d}  ({exact/n*100:4.1f}%)")
    print(f"correct, credited by tolerance  : {tol:5d}  ({tol/n*100:4.1f}%)"
          f"   (±{scoring.NUMERIC_TOLERANCE:.0%})")

    # ── accuracy by gold shape ───────────────────────────────────
    def shape(r):
        golds = scoring.split_gold(r.get("ground_truth"))
        if any(scoring.is_unanswerable(g) for g in golds):
            return "unanswerable"
        return "numeric" if any(scoring.number_variants(g) for g in golds) else "text"

    by_shape = {}
    for r in valid:
        s = by_shape.setdefault(shape(r), [0, 0])
        s[1] += 1
        s[0] += bool(r.get("is_correct"))
    print("\naccuracy by gold shape")
    for k, (c, t) in sorted(by_shape.items()):
        print(f"  {k:<14} {c/t*100:5.1f}%  ({c}/{t})")

    # ── malformed paths: the headroom for drop_malformed_paths ───
    n_paths = n_bad = 0
    for r in valid:
        for raw in r.get("raw_outputs") or []:
            n_paths += 1
            n_bad += not scoring.has_final_answer(raw)
    if n_paths:
        print(f"\nsampled paths            : {n_paths}")
        print(f"  no 'Final Answer:' line: {n_bad}  ({n_bad/n_paths*100:.1f}%)")

    # ── how the vote resolved ────────────────────────────────────
    modes = Counter(r.get("vote_mode") for r in valid if r.get("vote_mode"))
    if modes:
        print("\nvote mode")
        for m, c in modes.most_common():
            ok = sum(1 for r in valid if r.get("vote_mode") == m and r.get("is_correct"))
            print(f"  {str(m):<14} {c:5d} questions   {ok/max(c,1)*100:5.1f}% correct")

    agree = [r.get("agreement") for r in valid if r.get("agreement") is not None]
    if agree:
        print(f"\nmean path agreement : {sum(agree)/len(agree):.3f}")
    for label, keep in (("unanimous", lambda a: a >= 0.999), ("split (<0.6)", lambda a: a < 0.6)):
        sub = [r for r in valid if keep(r.get("agreement") or 0)]
        if sub:
            print(f"  {label:<14} {len(sub):5d}  "
                  f"({sum(r['is_correct'] for r in sub)/len(sub)*100:.1f}% correct)")

    wrong = [r for r in valid if not r.get("is_correct")][:10]
    print(f"\nfirst {len(wrong)} misses:")
    for r in wrong:
        print(f"  · {str(r['question'])[:62]:<62} pred={str(r['prediction'])[:16]:<16} "
              f"gold={str(r['ground_truth'])[:20]}")

## Cell 14 — Outputs

Download `results/` from the version's **Output** tab. Dropping those files into
`chartqapro/results/` in the repo makes them show up in the **ChartQAPro** tab of the
web app (`localhost:5173/#cqa`) — the dashboard reads `summarize.py`, which is why its
numbers match Cells 9–11 exactly.

In [ ]:
print(f"{RESULTS_DIR}\n")
for p in sorted(RESULTS_DIR.iterdir()):
    print(f"  {p.name:<28} {p.stat().st_size/1024:9.1f} KB")

if meta and not meta.get("complete"):
    print("\n◐  PARTIAL RUN — to continue, add this version's output as a data source and set:")
    print(f'    RESUME_FROM = "/kaggle/input/<notebook-output>/results/checkpoint_factoid.json"')